# PhysIQ – Task 1: Trajectory Prediction

Given a 2D physics scenario, predict the final (x, y) position of a target object after the simulation runs.

**Format:** `ANSWER: [x, y]`  
**Scoring:** Continuous 0–1 based on normalised Euclidean distance to ground truth.

In [ ]:
# ── Cell 1: Setup & imports ──────────────────────────────────────────────────
import sys, os, json, re
import numpy as np
import pandas as pd

# Allow importing physiq from project root (Kaggle: attach physiq as dataset)
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Kaggle_agi_bench'))

import kaggle_benchmarks as kbench
print('kaggle-benchmarks', kbench.__version__)

In [ ]:
# ── Cell 2: Physics engine & scenario library ────────────────────────────────
from physiq.engine import PhysIQWorld
from physiq.formats import build_prompt
from physiq.generation import generate_dataset, build_evaluation_dataframes, compute_ground_truth, validate_scenario
from physiq.scoring import score_trajectory, parse_coordinates
from physiq.templates.trajectory import TRAJECTORY_TEMPLATES

print(f'Trajectory templates: {len(TRAJECTORY_TEMPLATES)}')

In [ ]:
# ── Cell 3: Generate trajectory scenarios ───────────────────────────────────
from physiq.generation import _task_seed
from physiq.templates import SCENARIO_COUNTS

MASTER_SEED = 42
task_type = 'trajectory'
counts = SCENARIO_COUNTS[task_type]   # {easy:20, medium:20, hard:20}

task_seed = _task_seed(task_type, MASTER_SEED)
rng = np.random.RandomState(task_seed)

scenarios = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected = []
    attempt = 0
    while len(collected) < target and attempt < target * 3:
        seed = int(rng.randint(0, 2**31))
        template = TRAJECTORY_TEMPLATES[attempt % len(TRAJECTORY_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            s['_ground_truth'] = compute_ground_truth(s, task_type)
            collected.append(s)
    scenarios.extend(collected)
    print(f'  {difficulty}: {len(collected)}/{target}')

print(f'Total trajectory scenarios: {len(scenarios)}')

In [ ]:
# ── Cell 4: Build evaluation DataFrame ──────────────────────────────────────
rows = []
for s in scenarios:
    gt = s['_ground_truth']
    for fmt in ['json', 'ascii', 'nl']:
        rows.append({
            'scenario_id':   s['id'],
            'difficulty':    s['difficulty'],
            'representation': fmt,
            'prompt':        build_prompt(s, fmt, task_type),
            'ground_truth':  json.dumps(gt),
        })

df_trajectory = pd.DataFrame(rows)
print(df_trajectory.shape)
df_trajectory.head(2)

In [ ]:
# ── Cell 5: Scoring helpers ──────────────────────────────────────────────────
def _parse_answer(response: str):
    """Extract [x, y] from model response. Returns None if unparseable."""
    # Primary: ANSWER: [x, y]
    m = re.search(r'ANSWER\s*:\s*\[([\d.\-+eE\s,]+)\]', response, re.IGNORECASE)
    if m:
        try:
            nums = [float(x) for x in m.group(1).split(',')]
            if len(nums) == 2:
                return nums
        except ValueError:
            pass
    # Fallback: last bracketed pair in text
    pairs = re.findall(r'\[([\d.\-+eE\s,]+)\]', response)
    for p in reversed(pairs):
        try:
            nums = [float(x) for x in p.split(',')]
            if len(nums) == 2:
                return nums
        except ValueError:
            pass
    return None


def _score_trajectory_response(response: str, ground_truth_json: str) -> float:
    """Parse model response and return 0-1 trajectory score."""
    gt = json.loads(ground_truth_json)
    predicted = _parse_answer(response)
    if predicted is None:
        return 0.0
    return score_trajectory(predicted, gt['final_position'], gt['world_diagonal'])


# Quick sanity check
gt_sample = df_trajectory.iloc[0]['ground_truth']
gt_dict = json.loads(gt_sample)
fake_perfect = f"After careful simulation, ANSWER: {gt_dict['final_position']}"
print('Perfect answer score:', _score_trajectory_response(fake_perfect, gt_sample))

In [ ]:
# ── Cell 6: Task definition ──────────────────────────────────────────────────
@kbench.task(
    name='PhysIQ Trajectory Prediction',
    description=(
        'Given a 2D physics scenario (as JSON, ASCII art, or natural language), '
        'predict the final (x, y) position of the target object after the '
        'simulation runs. Format answer as: ANSWER: [x, y]. '
        'Tests genuine mental simulation, not equation recall.'
    ),
)
def physiq_trajectory(llm: kbench.LLMChat, prompt: str, ground_truth: str) -> float:
    """Predict final object position in 2D physics simulation."""
    response = llm.prompt(prompt)
    return _score_trajectory_response(response, ground_truth)

In [ ]:
# ── Cell 7: Dry-run sanity check (local only) ────────────────────────────────
# Verify the task signature matches the DataFrame columns.
required_cols = {'prompt', 'ground_truth'}
assert required_cols.issubset(df_trajectory.columns), (
    f'Missing columns: {required_cols - set(df_trajectory.columns)}'
)
print('DataFrame columns OK:', list(df_trajectory.columns))
print('Rows:', len(df_trajectory))
print('Difficulties:', df_trajectory.groupby('difficulty').size().to_dict())
print('Formats:', df_trajectory.groupby('representation').size().to_dict())

In [ ]:
# ── Cell 8: Evaluation run ───────────────────────────────────────────────────
# On Kaggle, kbench.llm is the competition model.
try:
    llm = kbench.llm
except AttributeError:
    print('No Kaggle LLM available (local run). Skipping evaluation.')
    llm = None

if llm is not None:
    results = []
    for _, row in df_trajectory.iterrows():
        run = physiq_trajectory.run(
            llm=llm,
            prompt=row['prompt'],
            ground_truth=row['ground_truth'],
        )
        results.append({
            'scenario_id':    row['scenario_id'],
            'difficulty':     row['difficulty'],
            'representation': row['representation'],
            'score':          run.result,
        })
    df_results = pd.DataFrame(results)
    os.makedirs('../outputs', exist_ok=True)
    df_results.to_csv('../outputs/task1_trajectory_results.csv', index=False)
    print('Mean score:', df_results['score'].mean())
    print(df_results.groupby(['difficulty', 'representation'])['score'].mean().unstack())

In [ ]:
# ── Cell 9: Results analysis ─────────────────────────────────────────────────
if llm is not None and 'df_results' in dir():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Score by difficulty
    df_results.groupby('difficulty')['score'].mean().plot(
        kind='bar', ax=axes[0], color=['#4CAF50', '#FF9800', '#F44336']
    )
    axes[0].set_title('Trajectory Score by Difficulty')
    axes[0].set_ylabel('Mean Score (0-1)')
    axes[0].set_ylim(0, 1)

    # Score by format
    df_results.groupby('representation')['score'].mean().plot(
        kind='bar', ax=axes[1], color=['#2196F3', '#9C27B0', '#00BCD4']
    )
    axes[1].set_title('Trajectory Score by Format')
    axes[1].set_ylabel('Mean Score (0-1)')
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../outputs/task1_trajectory_results.png', dpi=150)
    plt.show()

    # Format Robustness Score per scenario
    frs_rows = []
    for sid, grp in df_results.groupby('scenario_id'):
        scores = grp['score'].values
        mx = scores.max()
        frs = 1.0 - (mx - scores.min()) / mx if mx > 0 else 1.0
        frs_rows.append({'scenario_id': sid, 'frs': frs})
    frs_mean = pd.DataFrame(frs_rows)['frs'].mean()
    print(f'Format Robustness Score: {frs_mean:.3f}')

In [ ]:
# ── Cell 10: Choose this task for the leaderboard ────────────────────────────
%choose physiq_trajectory